# Day 94: Build an Eval Harness for a Deployed Model

**Layer:** Advanced


## Concept Overview

An evaluation harness sends standardized prompts to a deployed model endpoint and measures accuracy on a task (MMLU, HumanEval, etc.). Running evals continuously catches model regressions from system changes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Eval Harness Framework

Implement a minimal eval harness that sends multiple-choice questions and scores accuracy.


In [ ]:
import asyncio, json, numpy as np, time

class EvalHarness:
    def __init__(self, base_url="http://localhost:8000", model="default"):
        self.base_url = base_url
        self.model = model
        self.results = []

    async def evaluate_item(self, question, choices, correct_idx, sem):
        async with sem:
            # Simulate model scoring each choice
            # In production: send each choice as completion and score log-prob
            np.random.seed(hash(question) % 2**32)
            logprobs = np.random.randn(len(choices))
            # Inject some signal
            logprobs[correct_idx] += np.random.normal(0.5, 0.3)
            predicted = np.argmax(logprobs)
            return predicted == correct_idx

    async def run(self, questions, concurrency=10):
        sem = asyncio.Semaphore(concurrency)
        tasks = [self.evaluate_item(q["question"], q["choices"],
                                     q["answer"], sem) for q in questions]
        results = await asyncio.gather(*tasks)
        return sum(results) / len(results)

# Simulate MMLU-style questions
np.random.seed(42)
questions = [
    {"question": f"Q{i}", "choices": ["A","B","C","D"], "answer": np.random.randint(4)}
    for i in range(200)
]

harness = EvalHarness()
accuracy = asyncio.run(harness.run(questions))
print(f"Eval harness results:")
print(f"  Questions: {len(questions)}")
print(f"  Accuracy: {accuracy:.1%}")
print(f"  Random baseline: 25.0%")
print(f"  Signal above random: {(accuracy-0.25)/0.25:.0%}")
print()
print("Production eval: run on MMLU/HumanEval subsets after each deploy.")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- An evaluation harness sends standardized prompts to a deployed model endpoint and measures accuracy on a task (MMLU, HumanEval, etc.).
- An evaluation harness sends standardized prompts to a deployed model endpoint an....
- Day 94 implementation complete.
